# NB01 — Data Import, Quality Control & Preprocessing
**Pipeline:** Generalised Metabolite–Metagenomics Correlation Study  
**Outputs:** `preprocessed_data.pkl` → consumed by NB02–NB06

### Analysis overview
| Step | Analysis |
|---|---|
| 1–5 | Data import, duplicate removal, metadata harmonisation, alignment |
| 6–7 | Sample-level QC + Shannon diversity Kruskal-Wallis test |
| 8–10 | Feature QC, species name reduction, prevalence/variance filtering |
| 11 | CLR transform (species) · log10+median-centre (metabolites) |
| 12–13 | Separate PCA per modality (top-10 PCs, scree, biplot) |
| 14 | PCoA — Bray-Curtis (species raw) + Aitchison (CLR) |
| 15 | Mantel test — microbiome ↔ metabolome distance correlation |
| 16 | RDA — species/metabolites ~ CRC stage group |
| 17 | Post-transform distribution QC |
| 18–19 | Metabolite name mapping · save preprocessed_data.pkl |


## 1 · Imports & Configuration

In [1]:
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.decomposition import PCA as _PCA
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path(".").resolve()))

from utils import (
    DATA_DIR, RESULTS_DIR, INTER_DIR, FIG_DIR, TABLE_DIR,
    DATASETS_ALL, DATASET_PRIMARY, DATASET_SECONDARY,
    CRC_STAGE_ORDER, CRC_STAGE_NUMERIC,
    THREE_GROUP_ORDER, THREE_GROUP_MAP,
    PREVALENCE_SPE, PREVALENCE_MTB, FDR_THRESHOLD,
    PALETTE_STAGE6, PALETTE_3GROUP,
    load_all_datasets,
    harmonize_metadata, validate_sample_alignment,
    build_metabolite_name_map,
    reduce_species_names,
    compute_sample_qc, compute_feature_qc,
    qc_filter_species, qc_filter_metabolites,
    clr_transform, log10_transform,
    shannon_kruskal_test,
    bray_curtis_pcoa, aitchison_pcoa,
    mantel_test, rda_analysis,
    save_pickle, savefig,
)

warnings.filterwarnings("ignore")
%matplotlib inline
%config InlineBackend.figure_format = "retina"
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

for d in [INTER_DIR, FIG_DIR/"qc", FIG_DIR/"ordination", TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Imports OK")
print(f"  DATA_DIR    : {DATA_DIR}")
print(f"  Datasets    : {DATASETS_ALL}")
print(f"  Prevalence  : species >= {PREVALENCE_SPE:.0%}  |  metabolites >= {PREVALENCE_MTB:.0%}")


Imports OK
  DATA_DIR    : C:\Users\andna\Documents\KI\Data
  Datasets    : ['ERAWIJANTARI-GASTRIC-CANCER-2020', 'MARS-IBS-2020', 'WANG_ESRD_2020', 'YACHIDA-CRC-2019', 'SINHA_CRC_2016', 'Kim_adenomas_2020', 'KONG_EOCRC_2023', 'SINHA_VOGTMANN_SHOTGUN', 'BORENSTEIN_CRC']
  Prevalence  : species >= 10%  |  metabolites >= 15%


---
## 2 · Raw Data Import

In [2]:
data_raw = load_all_datasets(DATASETS_ALL, DATA_DIR)

summary_rows = []
for ds, d in data_raw.items():
    meta = d.get("metadata", pd.DataFrame())
    mtb  = d.get("mtb",      pd.DataFrame())
    spe  = d.get("species",  pd.DataFrame())
    summary_rows.append({
        "Dataset":     ds,
        "Samples":     len(meta),
        "Species":     spe.shape[1] if not spe.empty else 0,
        "Metabolites": mtb.shape[1] if not mtb.empty else 0,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))


Loaded 9 datasets: ['ERAWIJANTARI-GASTRIC-CANCER-2020', 'MARS-IBS-2020', 'WANG_ESRD_2020', 'YACHIDA-CRC-2019', 'SINHA_CRC_2016', 'Kim_adenomas_2020', 'KONG_EOCRC_2023', 'SINHA_VOGTMANN_SHOTGUN', 'BORENSTEIN_CRC']
                         Dataset  Samples  Species  Metabolites
ERAWIJANTARI-GASTRIC-CANCER-2020       96    48243          524
                   MARS-IBS-2020      444    11513           43
                  WANG_ESRD_2020      287    56961          276
                YACHIDA-CRC-2019      347    57702          450
                  SINHA_CRC_2016      131       86          530
               Kim_adenomas_2020      240      499          462
                 KONG_EOCRC_2023        0        0            0
          SINHA_VOGTMANN_SHOTGUN        0        0            0
                  BORENSTEIN_CRC        0        0            0


---
## 3 · ERAWIJANTARI-YACHIDA Duplicate Detection
Both datasets overlap in Japanese cohorts. Duplicate sample IDs are removed from ERAWIJANTARI.

In [3]:
era_key = "ERAWIJANTARI-GASTRIC-CANCER-2020"
yac_key = DATASET_PRIMARY

if era_key in data_raw and yac_key in data_raw:
    era_meta = data_raw[era_key].get("metadata", pd.DataFrame())
    yac_meta = data_raw[yac_key].get("metadata", pd.DataFrame())

    # Use publisher-provided shared-sample flags (IDs differ across datasets)
    era_flag = "Shared.w.YACHIDA_2019"
    yac_flag = "Shared.w.ERAWIJANTARI_2020"

    if era_flag in era_meta.columns:
        dup_era = set(era_meta.index[
            era_meta[era_flag].astype(str).str.upper() == "TRUE"])
    else:
        dup_era = set()

    if dup_era:
        print(f"  Removing {len(dup_era)} ERAWIJANTARI samples shared with YACHIDA")
        for key in ("metadata", "mtb", "species"):
            df = data_raw[era_key].get(key, pd.DataFrame())
            if not df.empty:
                data_raw[era_key][key] = df.drop(
                    index=[i for i in dup_era if i in df.index])
        print(f"  ERAWIJANTARI after removal: {len(data_raw[era_key]['metadata'])} samples")
    else:
        print("  No ERAWIJANTARI-YACHIDA duplicates detected.")

    # Report YACHIDA side (kept — YACHIDA is primary)
    if yac_flag in yac_meta.columns:
        n_yac_shared = yac_meta[yac_flag].astype(str).str.upper().eq("TRUE").sum()
        if n_yac_shared:
            print(f"  Note: {n_yac_shared} YACHIDA samples flagged as shared (kept)")


  Removing 53 ERAWIJANTARI samples shared with YACHIDA
  ERAWIJANTARI after removal: 43 samples
  Note: 53 YACHIDA samples flagged as shared (kept)


---
## 4 · Metadata Harmonisation

In [4]:
harmonized_meta = {}
for ds in data_raw:
    raw_meta = data_raw[ds].get("metadata", pd.DataFrame())
    harmonized_meta[ds] = (
        harmonize_metadata(raw_meta, ds) if not raw_meta.empty else pd.DataFrame())

ymd = harmonized_meta.get(DATASET_PRIMARY, pd.DataFrame())
if not ymd.empty:
    print("YACHIDA 6-level stage counts:")
    print(ymd["Stage.6"].value_counts().reindex(CRC_STAGE_ORDER).fillna(0).astype(int))
    print("\nYACHIDA 3-group counts:")
    print(ymd["Stage.3Group"].value_counts().reindex(THREE_GROUP_ORDER).fillna(0).astype(int))

# Confounder availability
conf_cols = ["Age", "BMI", "Sex", "Gender", "Smoking", "Alcohol"]
if not ymd.empty:
    avail   = [c for c in conf_cols if c in ymd.columns and ymd[c].notna().mean() > 0.3]
    missing = [c for c in conf_cols if c not in avail]
    print(f"\nAvailable confounders (>30% complete): {avail}")
    print(f"Missing/sparse confounders            : {missing}")


YACHIDA 6-level stage counts:
Stage.6
Healthy         127
Stage_0          27
HS               30
Stage_I_II       69
Stage_III_IV     54
MP               40
Name: count, dtype: int64

YACHIDA 3-group counts:
Stage.3Group
Healthy         127
Early_CRC        57
Advanced_CRC    163
Name: count, dtype: int64

Available confounders (>30% complete): ['Age', 'BMI', 'Gender', 'Alcohol']
Missing/sparse confounders            : ['Sex', 'Smoking']


---
## 5 · Sample Alignment Validation

In [5]:
alignment = {}
for ds in data_raw:
    meta = harmonized_meta.get(ds, pd.DataFrame())
    mtb  = data_raw[ds].get("mtb",     pd.DataFrame())
    spe  = data_raw[ds].get("species", pd.DataFrame())
    if meta.empty or mtb.empty or spe.empty:
        alignment[ds] = {"n_shared_samples": 0, "shared": set(),
                         "n_meta_only": 0, "n_mtb_only": 0, "n_species_only": 0}
        print(f"  {ds:<42}  SKIPPED (missing modality)")
        continue
    alignment[ds] = validate_sample_alignment(meta, mtb, spe)
    a = alignment[ds]
    print(f"  {ds:<42}  shared={a['n_shared_samples']:>4}  "
          f"meta_only={a['n_meta_only']:>3}  "
          f"mtb_only={a['n_mtb_only']:>3}  "
          f"spe_only={a['n_species_only']:>3}")


  ERAWIJANTARI-GASTRIC-CANCER-2020            shared=  43  meta_only=  0  mtb_only=  0  spe_only=  0
  MARS-IBS-2020                               shared= 444  meta_only=  0  mtb_only=  0  spe_only=  0
  WANG_ESRD_2020                              shared= 287  meta_only=  0  mtb_only=  0  spe_only=  0
  YACHIDA-CRC-2019                            shared= 347  meta_only=  0  mtb_only=  0  spe_only=  0
  SINHA_CRC_2016                              shared= 131  meta_only=  0  mtb_only=  0  spe_only=  0
  Kim_adenomas_2020                           shared= 240  meta_only=  0  mtb_only=  0  spe_only=  0
  KONG_EOCRC_2023                             SKIPPED (missing modality)
  SINHA_VOGTMANN_SHOTGUN                      SKIPPED (missing modality)
  BORENSTEIN_CRC                              SKIPPED (missing modality)


---
## 6 · Sample-Level QC
Shannon diversity, total metabolite signal, and detection rate per sample. Outlier samples (MetaPhlAn row sum >> 1) are flagged.

In [6]:
sample_qc_spe = {}
sample_qc_mtb = {}

for ds in data_raw:
    shared = list(alignment.get(ds, {}).get("shared", set()))
    if not shared:
        continue
    spe = data_raw[ds].get("species", pd.DataFrame())
    mtb = data_raw[ds].get("mtb",     pd.DataFrame())
    if not spe.empty:
        sp_shared = spe.loc[[s for s in shared if s in spe.index]]
        sample_qc_spe[ds] = compute_sample_qc(sp_shared, data_type="species")
    if not mtb.empty:
        mt_shared = mtb.loc[[s for s in shared if s in mtb.index]]
        sample_qc_mtb[ds] = compute_sample_qc(mt_shared, data_type="metabolite")

# MetaPhlAn row-sum check (YACHIDA) — should be ~1.0
yk = DATASET_PRIMARY
if yk in sample_qc_spe:
    rs  = sample_qc_spe[yk]["row_sum"]
    n_bad = (rs > 1.1).sum()
    print(f"YACHIDA species row-sum: mean={rs.mean():.4f}  min={rs.min():.4f}  max={rs.max():.4f}")
    if n_bad:
        print(f"  WARNING: {n_bad} samples have row_sum > 1.1 — check MetaPhlAn filtering level")
    else:
        print("  Row sums look correct (all <= 1.1)")


YACHIDA species row-sum: mean=1.0000  min=1.0000  max=1.0000
  Row sums look correct (all <= 1.1)


In [7]:
# Shannon diversity + detection rate boxplots (YACHIDA)
if DATASET_PRIMARY in sample_qc_spe and DATASET_PRIMARY in sample_qc_mtb:
    meta_pca = harmonized_meta[DATASET_PRIMARY]
    grp_col  = "Stage.3Group"
    qc_s = sample_qc_spe[DATASET_PRIMARY].join(meta_pca[[grp_col]], how="left")
    qc_m = sample_qc_mtb[DATASET_PRIMARY].join(meta_pca[[grp_col]], how="left")
    order = [g for g in THREE_GROUP_ORDER if g in qc_s[grp_col].values]
    pal   = {g: PALETTE_3GROUP[g] for g in order}

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, df_, col, title in zip(
        axes,
        [qc_s, qc_s, qc_m],
        ["shannon_diversity", "detection_rate", "total_signal"],
        ["Shannon Diversity (species)", "Species Detection Rate", "Metabolite Total Signal"]
    ):
        sns.boxplot(data=df_, x=grp_col, y=col, order=order,
                    palette=pal, ax=ax, showfliers=False)
        sns.stripplot(data=df_, x=grp_col, y=col, order=order,
                      color="black", alpha=0.3, size=2, ax=ax, jitter=True)
        ax.set_title(title, fontweight="bold")
        ax.tick_params(axis="x", rotation=20)
        ax.set_xlabel("")

    plt.suptitle(f"YACHIDA  |  n={len(qc_s)}  |  Sample-level QC",
                 fontweight="bold", y=1.02)
    plt.tight_layout()
    savefig(fig, "qc", "nb01_sample_qc.png")


Saved figure: C:\Users\andna\Documents\KI\Results\figures\qc\nb01_sample_qc.png


---
## 7 · Shannon Diversity Statistical Test
Kruskal-Wallis across Healthy / Early_CRC / Advanced_CRC, followed by pairwise Mann-Whitney U with BH correction.

In [8]:
yk = DATASET_PRIMARY
shannon_test_result = pd.DataFrame()

if yk in sample_qc_spe and not harmonized_meta.get(yk, pd.DataFrame()).empty:
    shannon_test_result = shannon_kruskal_test(
        qc_df     = sample_qc_spe[yk],
        meta      = harmonized_meta[yk],
        group_col = "Stage.3Group",
        groups    = THREE_GROUP_ORDER,
    )

    kw_stat = shannon_test_result.attrs.get("kruskal_stat", np.nan)
    kw_pval = shannon_test_result.attrs.get("kruskal_pval", np.nan)
    print(f"Kruskal-Wallis (Shannon diversity ~ Stage.3Group):")
    print(f"  H = {kw_stat:.3f},  p = {kw_pval:.4e}")
    if kw_pval < 0.05:
        print("  -> Significant difference in alpha diversity across groups")
    else:
        print("  -> No significant difference in alpha diversity")
    print("\nPairwise Mann-Whitney U (BH corrected):")
    if not shannon_test_result.empty:
        print(shannon_test_result[["group1","group2","n1","n2","pval_raw","qval"]].to_string(index=False))
        shannon_test_result.to_csv(TABLE_DIR / "shannon_diversity_test.csv", index=False)

    # Annotated violin plot with significance
    qc_s = sample_qc_spe[yk].join(harmonized_meta[yk][["Stage.3Group"]], how="left")
    order = [g for g in THREE_GROUP_ORDER if g in qc_s["Stage.3Group"].values]
    pal   = {g: PALETTE_3GROUP[g] for g in order}

    fig, ax = plt.subplots(figsize=(7, 5))
    sns.violinplot(data=qc_s, x="Stage.3Group", y="shannon_diversity",
                   order=order, palette=pal, ax=ax, inner=None, alpha=0.6)
    sns.boxplot(data=qc_s, x="Stage.3Group", y="shannon_diversity",
                order=order, palette=pal, ax=ax,
                width=0.2, showfliers=False, boxprops={"alpha": 0.9})

    # Add KW p-value annotation
    ax.text(0.98, 0.98, f"KW p = {kw_pval:.3e}",
            transform=ax.transAxes, ha="right", va="top",
            fontsize=9, style="italic",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

    ax.set_xlabel("CRC Group")
    ax.set_ylabel("Shannon Diversity (H')")
    ax.set_title("Species Alpha Diversity — YACHIDA", fontweight="bold")
    plt.tight_layout()
    savefig(fig, "qc", "nb01_shannon_diversity.png")


Kruskal-Wallis (Shannon diversity ~ Stage.3Group):
  H = 1.638,  p = 4.4087e-01
  -> No significant difference in alpha diversity

Pairwise Mann-Whitney U (BH corrected):
   group1       group2  n1  n2  pval_raw     qval
  Healthy    Early_CRC 127  57  0.668610 0.672252
  Healthy Advanced_CRC 127 163  0.191229 0.573688
Early_CRC Advanced_CRC  57 163  0.672252 0.672252
Saved figure: C:\Users\andna\Documents\KI\Results\figures\qc\nb01_shannon_diversity.png


---
## 8 · Feature-Level QC & Detection Rate Histograms

In [9]:
ds = DATASET_PRIMARY
if ds in data_raw:
    shared    = list(alignment[ds]["shared"])
    spe       = data_raw[ds].get("species", pd.DataFrame())
    mtb       = data_raw[ds].get("mtb",     pd.DataFrame())
    sp_shared = spe.loc[[s for s in shared if s in spe.index]]
    mt_shared = mtb.loc[[s for s in shared if s in mtb.index]]
    fqc_spe   = compute_feature_qc(sp_shared)
    fqc_mtb   = compute_feature_qc(mt_shared)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(fqc_spe["detection_rate"], bins=50,
                 color="#4CAF50", edgecolor="white", lw=0.4)
    axes[0].axvline(PREVALENCE_SPE, color="red", ls="--",
                    label=f"threshold ({PREVALENCE_SPE:.0%})")
    axes[0].set_xlabel("Detection rate (fraction of samples > 0)")
    axes[0].set_title("Species — feature detection rates (raw)")
    axes[0].legend(fontsize=8)

    axes[1].hist(fqc_mtb["detection_rate"], bins=50,
                 color="#2196F3", edgecolor="white", lw=0.4)
    axes[1].axvline(PREVALENCE_MTB, color="red", ls="--",
                    label=f"threshold ({PREVALENCE_MTB:.0%})")
    axes[1].set_xlabel("Detection rate (fraction of samples > 0)")
    axes[1].set_title("Metabolites — feature detection rates (raw)")
    axes[1].legend(fontsize=8)

    plt.suptitle("YACHIDA — Feature Detection Rates (before QC)", fontweight="bold")
    plt.tight_layout()
    savefig(fig, "qc", "nb01_feature_detection_rates.png")

    n_spe_pass = (fqc_spe["detection_rate"] >= PREVALENCE_SPE).sum()
    n_mtb_pass = (fqc_mtb["detection_rate"] >= PREVALENCE_MTB).sum()
    print(f"YACHIDA species  passing >= {PREVALENCE_SPE:.0%}: {n_spe_pass} / {len(fqc_spe)}")
    print(f"YACHIDA metabolites passing >= {PREVALENCE_MTB:.0%}: {n_mtb_pass} / {len(fqc_mtb)}")


Saved figure: C:\Users\andna\Documents\KI\Results\figures\qc\nb01_feature_detection_rates.png
YACHIDA species  passing >= 10%: 17692 / 57702
YACHIDA metabolites passing >= 15%: 277 / 450


---
## 9 · Species Name Reduction
Collapse full MetaPhlAn taxonomy strings to species-level short names. Sum duplicate names.

In [10]:
species_reduced  = {}
species_name_map = {}

for ds in data_raw:
    spe    = data_raw[ds].get("species", pd.DataFrame())
    shared = list(alignment.get(ds, {}).get("shared", set()))
    if spe.empty or not shared:
        continue
    sp_aligned = spe.loc[[s for s in shared if s in spe.index]]
    reduced, nm = reduce_species_names(sp_aligned)
    species_reduced[ds]  = reduced
    species_name_map[ds] = nm
    n_collapsed = sum(1 for v in nm.values() if len(v) > 1)
    print(f"  {ds:<42}  {sp_aligned.shape[1]:>4} -> {reduced.shape[1]:>4} "
          f"({n_collapsed} names collapsed)")


  ERAWIJANTARI-GASTRIC-CANCER-2020            48243 -> 48243 (0 names collapsed)
  MARS-IBS-2020                               11513 -> 11513 (0 names collapsed)
  WANG_ESRD_2020                              56961 -> 56961 (0 names collapsed)
  YACHIDA-CRC-2019                            57702 -> 57702 (0 names collapsed)
  SINHA_CRC_2016                                86 ->   67 (1 names collapsed)
  Kim_adenomas_2020                            499 ->  444 (1 names collapsed)


---
## 10 · QC Filtering
Prevalence + near-zero-variance filtering. No pathway-based filtering applied.

In [11]:
species_filtered = {}
mtb_filtered     = {}

for ds in data_raw:
    meta   = harmonized_meta.get(ds, pd.DataFrame())
    mtb    = data_raw[ds].get("mtb", pd.DataFrame())
    spe_r  = species_reduced.get(ds)
    shared = list(alignment.get(ds, {}).get("shared", set()))
    if not shared or meta.empty or mtb.empty or spe_r is None:
        continue

    mt_aligned = mtb.loc[[s for s in shared if s in mtb.index]]
    sp_qc = qc_filter_species(spe_r)
    mt_qc = qc_filter_metabolites(mt_aligned)

    species_filtered[ds] = sp_qc
    mtb_filtered[ds]     = mt_qc

    sp_sparsity = (sp_qc == 0).values.mean()
    flag = "  WARNING: high sparsity" if sp_sparsity > 0.8 else ""
    print(f"  {ds:<42}  "
          f"species: {spe_r.shape[1]:>4} -> {sp_qc.shape[1]:>4}  "
          f"metabolites: {mt_aligned.shape[1]:>5} -> {mt_qc.shape[1]:>5}  "
          f"sparsity={sp_sparsity:.1%}{flag}")

print(f"\nDatasets retained after QC: {list(species_filtered.keys())}")


  ERAWIJANTARI-GASTRIC-CANCER-2020            species: 48243 -> 2396  metabolites:   524 ->   190  sparsity=16.8%
  MARS-IBS-2020                               species: 11513 -> 2741  metabolites:    43 ->    38  sparsity=51.9%
  WANG_ESRD_2020                              species: 56961 -> 4171  metabolites:   276 ->   248  sparsity=20.0%
  YACHIDA-CRC-2019                            species: 57702 -> 4392  metabolites:   450 ->   249  sparsity=27.4%
  SINHA_CRC_2016                              species:   67 ->   67  metabolites:   530 ->   477  sparsity=54.4%
  Kim_adenomas_2020                           species:  444 ->  152  metabolites:   462 ->   415  sparsity=60.0%

Datasets retained after QC: ['ERAWIJANTARI-GASTRIC-CANCER-2020', 'MARS-IBS-2020', 'WANG_ESRD_2020', 'YACHIDA-CRC-2019', 'SINHA_CRC_2016', 'Kim_adenomas_2020']


---
## 11 · Transformations
**Species:** CLR (pseudocount = 1e-4).  
**Metabolites:** half-min impute → log10(x+1) → sample-wise median centering.

In [12]:
species_clr = {}
mtb_log10   = {}

for ds in species_filtered:
    species_clr[ds] = clr_transform(species_filtered[ds])
    mtb_log10[ds]   = log10_transform(mtb_filtered[ds])

yk = DATASET_PRIMARY
print(f"YACHIDA species CLR      : {species_clr.get(yk, pd.DataFrame()).shape}")
print(f"YACHIDA metabolites log10: {mtb_log10.get(yk, pd.DataFrame()).shape}")

if yk in species_clr:
    vals = species_clr[yk].values
    pct  = (vals < -10).mean()
    print(f"\nCLR sanity: min={vals.min():.2f}  max={vals.max():.2f}  frac<-10: {pct:.2%}")
    if pct > 0.05:
        print("  WARNING: >5% CLR values below -10 — consider increasing pseudocount")

if yk in mtb_log10:
    row_meds = np.nanmedian(mtb_log10[yk].values, axis=1)
    print(f"Metabolite row-median post-transform: mean={row_meds.mean():.4f} (should be ~0)")


YACHIDA species CLR      : (347, 4392)
YACHIDA metabolites log10: (347, 249)

CLR sanity: min=-4.60  max=10.53  frac<-10: 0.00%
Metabolite row-median post-transform: mean=0.0000 (should be ~0)


---
## 12 · PCA — Separate per Modality (top 10 PCs)
Post-CLR/log10 PCA. Scree plot + PC1/PC2 biplot per modality, coloured by Stage.3Group.

In [13]:
ds = DATASET_PRIMARY
if ds in species_clr and not species_clr[ds].empty:
    meta_sub = harmonized_meta[ds].loc[species_clr[ds].index]
    grp      = meta_sub["Stage.3Group"].fillna("Unknown")

    n_pcs = min(10, species_clr[ds].shape[1] - 1, species_clr[ds].shape[0] - 1)

    for mod, mat, label in [
        ("species",     species_clr[ds].values,                "Species (CLR)"),
        ("metabolites", mtb_log10[ds].reindex(species_clr[ds].index).values, "Metabolites (log10+centre)"),
    ]:
        n_pcs_mod = min(n_pcs, mat.shape[1] - 1)
        pca = _PCA(n_components=n_pcs_mod, random_state=42)
        coords = pca.fit_transform(mat)
        ev     = pca.explained_variance_ratio_

        # ---- Scree plot ----
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        axes[0].bar(range(1, len(ev) + 1), ev * 100, color="#5C6BC0", edgecolor="white")
        axes[0].plot(range(1, len(ev) + 1), np.cumsum(ev) * 100,
                     "o-", color="#F44336", label="Cumulative")
        axes[0].set_xlabel("Principal Component")
        axes[0].set_ylabel("Variance Explained (%)")
        axes[0].set_title(f"{label} — Scree Plot (top {n_pcs_mod} PCs)")
        axes[0].legend(fontsize=8)

        # ---- PC1 vs PC2 biplot ----
        for g in THREE_GROUP_ORDER + ["Unknown"]:
            mask = (grp == g).values
            if not mask.any():
                continue
            axes[1].scatter(coords[mask, 0], coords[mask, 1],
                            c=PALETTE_3GROUP.get(g, "#999"), label=g,
                            s=22, alpha=0.75, edgecolors="none")
        axes[1].set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
        axes[1].set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")
        axes[1].set_title(f"{label} — PC1 vs PC2")
        axes[1].legend(fontsize=8)

        plt.suptitle(f"YACHIDA  n={len(grp)}  |  Post-Transform PCA ({mod})",
                     fontweight="bold", y=1.02)
        plt.tight_layout()
        savefig(fig, "ordination", f"nb01_pca_{mod}.png")

        # ---- Top-10 PC contribution table ----
        ev_df = pd.DataFrame({
            "PC":         [f"PC{i+1}" for i in range(len(ev))],
            "var_pct":    ev * 100,
            "cum_var_pct": np.cumsum(ev) * 100,
        })
        print(f"\n{label} — variance explained per PC:")
        print(ev_df.to_string(index=False))
        ev_df.to_csv(TABLE_DIR / f"pca_variance_{mod}.csv", index=False)


Saved figure: C:\Users\andna\Documents\KI\Results\figures\ordination\nb01_pca_species.png

Species (CLR) — variance explained per PC:
  PC  var_pct  cum_var_pct
 PC1 6.660666     6.660666
 PC2 3.766677    10.427343
 PC3 3.243290    13.670633
 PC4 2.695655    16.366288
 PC5 1.715720    18.082008
 PC6 1.448917    19.530925
 PC7 1.336345    20.867270
 PC8 1.274359    22.141629
 PC9 1.132056    23.273685
PC10 0.942124    24.215808
Saved figure: C:\Users\andna\Documents\KI\Results\figures\ordination\nb01_pca_metabolites.png

Metabolites (log10+centre) — variance explained per PC:
  PC   var_pct  cum_var_pct
 PC1 12.217192    12.217192
 PC2  7.035098    19.252290
 PC3  6.042681    25.294971
 PC4  4.678464    29.973435
 PC5  3.985686    33.959121
 PC6  2.872477    36.831598
 PC7  2.301111    39.132709
 PC8  2.266030    41.398739
 PC9  1.856271    43.255009
PC10  1.735121    44.990130


---
## 13 · PCoA — Bray-Curtis (species) & Aitchison (CLR)
PCoA with Bray-Curtis uses raw relative abundances (pre-CLR).  
Aitchison PCoA uses CLR-transformed data (Euclidean distance in CLR space).

In [14]:
ds = DATASET_PRIMARY
pcoa_results = {}

if ds in species_filtered and not species_filtered[ds].empty:
    meta_sub = harmonized_meta[ds].loc[species_filtered[ds].index]
    grp      = meta_sub["Stage.3Group"].fillna("Unknown")

    # Bray-Curtis PCoA: on raw relative abundances
    print("Computing Bray-Curtis PCoA...")
    bc_pcoa = bray_curtis_pcoa(species_filtered[ds], n_components=10)
    pcoa_results["bray_curtis"] = bc_pcoa

    # Aitchison PCoA: on CLR-transformed data
    print("Computing Aitchison PCoA...")
    ait_pcoa = aitchison_pcoa(species_clr[ds], n_components=10)
    pcoa_results["aitchison"] = ait_pcoa

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, pcoa_res, title, dist_label in zip(
        axes,
        [bc_pcoa, ait_pcoa],
        ["Bray-Curtis PCoA", "Aitchison PCoA (CLR)"],
        ["Bray-Curtis", "Aitchison"],
    ):
        coords = pcoa_res["coords"]
        ev     = pcoa_res["var_explained"]
        for g in THREE_GROUP_ORDER + ["Unknown"]:
            mask = (grp == g).values
            if not mask.any():
                continue
            ax.scatter(coords.iloc[mask, 0], coords.iloc[mask, 1],
                       c=PALETTE_3GROUP.get(g, "#999"), label=g,
                       s=22, alpha=0.75, edgecolors="none")
        ax.set_xlabel(f"PCo1 ({ev[0]*100:.1f}%)")
        ax.set_ylabel(f"PCo2 ({ev[1]*100:.1f}%)" if len(ev) > 1 else "PCo2")
        ax.set_title(title, fontweight="bold")
        ax.legend(fontsize=8)

    plt.suptitle("YACHIDA — Microbial Beta Diversity", fontweight="bold", y=1.02)
    plt.tight_layout()
    savefig(fig, "ordination", "nb01_pcoa_species.png")

    print(f"Bray-Curtis PCo1-PCo2 explains: "
          f"{bc_pcoa['var_explained'][:2].sum()*100:.1f}%")
    print(f"Aitchison PCo1-PCo2 explains  : "
          f"{ait_pcoa['var_explained'][:2].sum()*100:.1f}%")


Computing Bray-Curtis PCoA...
Computing Aitchison PCoA...
Saved figure: C:\Users\andna\Documents\KI\Results\figures\ordination\nb01_pcoa_species.png
Bray-Curtis PCo1-PCo2 explains: 20.2%
Aitchison PCo1-PCo2 explains  : 10.4%


---
## 14 · Mantel Test — Microbiome ↔ Metabolome Distance Correlation
Tests whether samples that are similar in microbial composition are also similar in
metabolite profile. Uses Pearson r of upper-triangle distance matrices.

- Microbiome: Aitchison distance (CLR Euclidean)
- Metabolome: Euclidean distance on log10-centred data

*Note: 999 permutations (~1–2 min).*

In [15]:
ds = DATASET_PRIMARY
mantel_results = {}

if ds in species_clr and ds in mtb_log10 and "aitchison" in pcoa_results:
    from scipy.spatial.distance import pdist, squareform

    common_idx = species_clr[ds].index.intersection(mtb_log10[ds].index)
    D_spe = pcoa_results["aitchison"]["dist_mat"]
    D_mtb = squareform(pdist(mtb_log10[ds].loc[common_idx].values, metric="euclidean"))

    # Reorder D_spe to match common_idx if needed
    spe_idx_map = {s: i for i, s in enumerate(species_clr[ds].index)}
    common_positions = [spe_idx_map[s] for s in common_idx if s in spe_idx_map]
    D_spe_common = D_spe[np.ix_(common_positions, common_positions)]

    print("Running Mantel test (999 permutations)...")
    r_mantel, p_mantel = mantel_test(D_spe_common, D_mtb, n_perm=999, random_state=42)
    mantel_results["microbiome_vs_metabolome"] = {"r": r_mantel, "p": p_mantel, "n": len(common_idx)}

    print(f"\nMantel test — Aitchison (microbiome) vs Euclidean (metabolome):")
    print(f"  r = {r_mantel:.4f},  p = {p_mantel:.4f} (999 permutations)")
    print(f"  n = {len(common_idx)} shared samples")
    if p_mantel < 0.05:
        print("  -> Significant microbiome-metabolome co-structure")
    else:
        print("  -> No significant microbiome-metabolome distance correlation")

    pd.DataFrame([{
        "comparison": "microbiome_vs_metabolome",
        "distance_microbiome": "Aitchison",
        "distance_metabolome": "Euclidean",
        "r": r_mantel, "p_value": p_mantel, "n": len(common_idx)
    }]).to_csv(TABLE_DIR / "mantel_test_results.csv", index=False)
else:
    print("Skipping Mantel test — PCoA or transform data missing.")


Running Mantel test (999 permutations)...

Mantel test — Aitchison (microbiome) vs Euclidean (metabolome):
  r = 0.2372,  p = 0.0010 (999 permutations)
  n = 347 shared samples
  -> Significant microbiome-metabolome co-structure


---
## 15 · RDA — Constrained Ordination (Species / Metabolites ~ CRC Stage)
Redundancy Analysis regresses each feature on one-hot encoded Stage.3Group,
then runs PCA on fitted values. R² indicates how much of the total feature
variance is explained by CRC stage.

In [16]:
ds = DATASET_PRIMARY
rda_results = {}

if ds in species_clr and not harmonized_meta.get(ds, pd.DataFrame()).empty:
    meta_sub = harmonized_meta[ds].loc[species_clr[ds].index]
    grp      = meta_sub["Stage.3Group"].dropna()
    common   = species_clr[ds].index.intersection(grp.index)

    # One-hot encode Stage.3Group as explanatory variable
    X_rda = pd.get_dummies(grp.loc[common], prefix="grp", drop_first=True).astype(float)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, Y_df, mod_label in [
        (axes[0], species_clr[ds].loc[common],
         "Species (CLR)"),
        (axes[1], mtb_log10[ds].reindex(common),
         "Metabolites (log10)"),
    ]:
        res = rda_analysis(Y_df, X_rda, n_components=2)
        rda_results[mod_label] = res
        scores = res["rda_scores"]

        for g in THREE_GROUP_ORDER:
            idx_g = grp.loc[common][grp.loc[common] == g].index
            ax.scatter(
                scores.reindex(idx_g).iloc[:, 0],
                scores.reindex(idx_g).iloc[:, 1] if scores.shape[1] > 1
                else np.zeros(len(idx_g)),
                c=PALETTE_3GROUP.get(g, "#999"), label=g, s=22, alpha=0.75
            )
        ax.set_xlabel("RDA1")
        ax.set_ylabel("RDA2")
        ax.set_title(
            f"{mod_label}\n"
            f"R² = {res['r2_total']:.3f}  R²adj = {res['r2_adjusted']:.3f}",
            fontweight="bold"
        )
        ax.legend(fontsize=8)
        print(f"RDA {mod_label}: R² = {res['r2_total']:.3f}, R²adj = {res['r2_adjusted']:.3f}")

    plt.suptitle("YACHIDA — RDA (Species/Metabolites ~ CRC Stage Group)",
                 fontweight="bold", y=1.02)
    plt.tight_layout()
    savefig(fig, "ordination", "nb01_rda.png")

    # Save RDA summary
    rda_summary = pd.DataFrame([
        {"modality": k, "r2": v["r2_total"], "r2_adj": v["r2_adjusted"],
         "n_samples": v["n_samples"], "n_predictors": v["n_predictors"]}
        for k, v in rda_results.items()
    ])
    rda_summary.to_csv(TABLE_DIR / "rda_summary.csv", index=False)
    print(rda_summary.to_string(index=False))


RDA Species (CLR): R² = 0.008, R²adj = 0.002
RDA Metabolites (log10): R² = 0.012, R²adj = 0.006
Saved figure: C:\Users\andna\Documents\KI\Results\figures\ordination\nb01_rda.png
           modality       r2   r2_adj  n_samples  n_predictors
      Species (CLR) 0.007607 0.001837        347             2
Metabolites (log10) 0.011591 0.005844        347             2


---
## 16 · Post-Transform Distribution QC

In [17]:
ds = DATASET_PRIMARY
if ds in species_clr and ds in mtb_log10:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    sp_med = species_clr[ds].median(axis=0)
    axes[0].hist(sp_med, bins=60, color="#4CAF50", edgecolor="white", lw=0.3)
    axes[0].axvline(0, color="red", ls="--", lw=1, label="zero")
    axes[0].set_xlabel("Per-feature median CLR")
    axes[0].set_title("Species — CLR per-feature medians")
    axes[0].legend(fontsize=8)

    mt_med = mtb_log10[ds].median(axis=0)
    axes[1].hist(mt_med, bins=60, color="#2196F3", edgecolor="white", lw=0.3)
    axes[1].axvline(0, color="red", ls="--", lw=1, label="zero")
    axes[1].set_xlabel("Per-feature median log10 (centred)")
    axes[1].set_title("Metabolites — log10 centred per-feature medians")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    savefig(fig, "qc", "nb01_feature_distributions_post_transform.png")


Saved figure: C:\Users\andna\Documents\KI\Results\figures\qc\nb01_feature_distributions_post_transform.png


---
## 17 · Metabolite Name Mapping

In [18]:
name_maps = {}
for ds in data_raw:
    mm = data_raw[ds].get("mtb.map", pd.DataFrame())
    name_maps[ds] = build_metabolite_name_map(mm)
    print(f"  {ds:<42}  {len(name_maps[ds]):>4} KEGG -> name entries")

yk_map = name_maps.get(DATASET_PRIMARY, {})
print(f"\nYACHIDA sample entries (first 8):")
for kid, name in list(yk_map.items())[:8]:
    print(f"  {kid}  ->  {name}")


  ERAWIJANTARI-GASTRIC-CANCER-2020             524 KEGG -> name entries
  MARS-IBS-2020                                 43 KEGG -> name entries
  WANG_ESRD_2020                               276 KEGG -> name entries
  YACHIDA-CRC-2019                             450 KEGG -> name entries
  SINHA_CRC_2016                               530 KEGG -> name entries
  Kim_adenomas_2020                            462 KEGG -> name entries
  KONG_EOCRC_2023                                0 KEGG -> name entries
  SINHA_VOGTMANN_SHOTGUN                         0 KEGG -> name entries
  BORENSTEIN_CRC                                 0 KEGG -> name entries

YACHIDA sample entries (first 8):
  C03173_(Methylthio)acetate  ->  C03173
  C03969_1-Amino-1-cyclopentanecarboxylate  ->  C03969
  C01234_1-Aminocyclopropane-1-carboxylate  ->  C01234
  C11118_1-Methyl-2-pyrrolidinone  ->  C11118
  C02494_1-Methyladenosine  ->  C02494
  C05127_1-Methylhistamine  ->  C05127
  C02918_1-Methylnicotinamide  ->  C02918


---
## 18 · Save Preprocessed Data

In [19]:
preprocessed = {
    "data_raw":          data_raw,
    "harmonized_meta":   harmonized_meta,
    "species_filtered":  species_filtered,
    "mtb_filtered":      mtb_filtered,
    "species_clr":       species_clr,
    "mtb_log10":         mtb_log10,
    "name_maps":         name_maps,
    "alignment":         alignment,
    "species_name_map":  species_name_map,
    "sample_qc_spe":     sample_qc_spe,
    "sample_qc_mtb":     sample_qc_mtb,
    "shannon_test":      shannon_test_result,
    "pcoa_results":      pcoa_results,
    "mantel_results":    mantel_results,
    "rda_results":       rda_results,
}

save_pickle(preprocessed, INTER_DIR / "preprocessed_data.pkl")

print("\n-- NB01 complete ---------------------------------------------------")
yk_spe = species_clr.get(DATASET_PRIMARY, pd.DataFrame())
yk_mtb = mtb_log10.get(DATASET_PRIMARY,  pd.DataFrame())
print(f"  YACHIDA species  (CLR)           : {yk_spe.shape[1]:>4} features  {yk_spe.shape[0]} samples")
print(f"  YACHIDA metabolites (log10+ctr)  : {yk_mtb.shape[1]:>5} features  {yk_mtb.shape[0]} samples")
print(f"  Shannon diversity KW pval        : "
      f"{shannon_test_result.attrs.get('kruskal_pval', 'N/A'):.4e}" if not shannon_test_result.empty else "  Shannon: N/A")
print(f"  Mantel r / p                     : "
      f"{mantel_results.get('microbiome_vs_metabolome', {}).get('r', 'N/A'):.3f} / "
      f"{mantel_results.get('microbiome_vs_metabolome', {}).get('p', 'N/A'):.4f}" if mantel_results else "  Mantel: N/A")
print("  -> Run NB02 (02_association_analysis.ipynb) next.")


Saved: C:\Users\andna\Documents\KI\Results\intermediate\preprocessed_data.pkl

-- NB01 complete ---------------------------------------------------
  YACHIDA species  (CLR)           : 4392 features  347 samples
  YACHIDA metabolites (log10+ctr)  :   249 features  347 samples
  Shannon diversity KW pval        : 4.4087e-01
  Mantel r / p                     : 0.237 / 0.0010
  -> Run NB02 (02_association_analysis.ipynb) next.
